# 03. Генерация benchmark-предсказаний

Блокнот воспроизводит финальный inference-пайплайн и создаёт `answer.csv`.
Тяжёлые артефакты рассчитываются только при отсутствии кэша.

Перед первым запуском необходимо выполнить `02_pipeline_development.ipynb`:
он обучает и сохраняет финальный агрегатор.

## 1. Окружение и данные

In [ ]:
import gc
import os
import sys
from pathlib import Path

os.environ.pop("SSLKEYLOGFILE", None)

import joblib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import integer_ids, prepare_coordinates
from src.fusion import FEATURE_NAMES, rank_with_model
from src.retrieval import (
    load_or_build_global_semantic,
    load_or_build_local_semantic,
    load_or_build_local_tfidf,
    load_or_build_nearby_semantic,
    load_or_build_tfidf,
    load_or_encode_embeddings,
)
from src.submission import (
    build_answer,
    positions_to_predictions,
    save_answer,
    validate_answer,
)
from src.text_features import (
    build_lexical_item_text,
    build_query_text,
    build_semantic_item_text,
)

DATA_DIR = PROJECT_ROOT / "dataset"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
BENCHMARK_ARTIFACTS_DIR = ARTIFACTS_DIR / "benchmark"
BENCHMARK_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
train = pd.read_parquet(DATA_DIR / "train.parquet")
benchmark_queries = pd.read_parquet(
    DATA_DIR / "benchmark_queries.parquet"
).reset_index(drop=True)
benchmark_items = pd.read_parquet(
    DATA_DIR / "benchmark_items.parquet"
).reset_index(drop=True)

assert benchmark_queries["query_id"].is_unique
assert benchmark_items["item_id"].is_unique
assert benchmark_queries["query_id"].astype(str).str.len().eq(16).all()
assert benchmark_items["item_id"].astype(str).str.len().eq(16).all()

train.shape, benchmark_queries.shape, benchmark_items.shape

## 2. Признаки и координаты

In [ ]:
location_reference_items = pd.concat(
    [
        train[["item_location_id", "item_latitude", "item_longitude"]],
        benchmark_items[["item_location_id", "item_latitude", "item_longitude"]],
    ],
    ignore_index=True,
).drop_duplicates()

(
    benchmark_queries,
    item_latitudes,
    item_longitudes,
    query_latitudes,
    query_longitudes,
) = prepare_coordinates(
    benchmark_queries,
    benchmark_items,
    location_reference_items,
    train,
)

item_ids = benchmark_items["item_id"].astype(str).to_numpy(dtype="U16")
query_ids = benchmark_queries["query_id"].astype(str).to_numpy(dtype="U16")
item_location_ids = integer_ids(benchmark_items["item_location_id"])
item_category_ids = integer_ids(benchmark_items["item_category_id"])
query_location_ids = integer_ids(benchmark_queries["search_location_id"])
query_category_ids = integer_ids(benchmark_queries["search_category"])

arrays_to_save = {
    "item_ids.npy": item_ids,
    "query_ids.npy": query_ids,
    "item_location_ids.npy": item_location_ids,
    "item_category_ids.npy": item_category_ids,
    "query_location_ids.npy": query_location_ids,
    "query_category_ids.npy": query_category_ids,
    "item_latitudes.npy": item_latitudes,
    "item_longitudes.npy": item_longitudes,
    "query_latitudes.npy": query_latitudes,
    "query_longitudes.npy": query_longitudes,
}
for filename, values in arrays_to_save.items():
    np.save(BENCHMARK_ARTIFACTS_DIR / filename, values)

pd.Series({
    "item_coordinate_coverage": np.mean(
        np.isfinite(item_latitudes) & np.isfinite(item_longitudes)
    ),
    "query_coordinate_coverage": np.mean(
        np.isfinite(query_latitudes) & np.isfinite(query_longitudes)
    ),
})

In [ ]:
query_texts = build_query_text(benchmark_queries)
lexical_item_texts = build_lexical_item_text(benchmark_items)
semantic_item_texts = build_semantic_item_text(benchmark_items)

del train, location_reference_items
_ = gc.collect()

## 3. TF-IDF и E5-эмбеддинги

In [ ]:
tfidf_vectorizer, tfidf_item_matrix, tfidf_query_matrix = load_or_build_tfidf(
    lexical_item_texts,
    query_texts,
    BENCHMARK_ARTIFACTS_DIR / "tfidf_vectorizer.joblib",
    BENCHMARK_ARTIFACTS_DIR / "tfidf_item_matrix.npz",
    BENCHMARK_ARTIFACTS_DIR / "tfidf_query_matrix.npz",
)

tfidf_item_matrix.shape, tfidf_query_matrix.shape

In [ ]:
item_embeddings, query_embeddings = load_or_encode_embeddings(
    semantic_item_texts,
    query_texts,
    BENCHMARK_ARTIFACTS_DIR / "item_embeddings_e5_small.npy",
    BENCHMARK_ARTIFACTS_DIR / "query_embeddings_e5_small.npy",
)

item_embeddings.shape, query_embeddings.shape

## 4. Четыре retrieval-канала

In [ ]:
global_e5, global_e5_scores = load_or_build_global_semantic(
    item_embeddings,
    query_embeddings,
    BENCHMARK_ARTIFACTS_DIR / "global_e5_top100_indices.npy",
    BENCHMARK_ARTIFACTS_DIR / "global_e5_top100_scores.npy",
    BENCHMARK_ARTIFACTS_DIR / "global_e5_hnsw_m16.faiss",
    top_k=100,
)

In [ ]:
local_e5, local_e5_scores = load_or_build_local_semantic(
    item_embeddings,
    query_embeddings,
    item_location_ids,
    item_category_ids,
    query_location_ids,
    query_category_ids,
    BENCHMARK_ARTIFACTS_DIR / "local_semantic_exact_top40_indices.npy",
    BENCHMARK_ARTIFACTS_DIR / "local_semantic_exact_top40_scores.npy",
    top_k=40,
)

local_tfidf, local_tfidf_scores = load_or_build_local_tfidf(
    tfidf_item_matrix,
    tfidf_query_matrix,
    item_location_ids,
    item_category_ids,
    query_location_ids,
    query_category_ids,
    BENCHMARK_ARTIFACTS_DIR / "local_tfidf_top40_indices.npy",
    BENCHMARK_ARTIFACTS_DIR / "local_tfidf_top40_scores.npy",
    top_k=40,
)

In [ ]:
nearby_e5 = load_or_build_nearby_semantic(
    item_embeddings,
    query_embeddings,
    item_location_ids,
    item_category_ids,
    query_location_ids,
    query_category_ids,
    item_latitudes,
    item_longitudes,
    query_latitudes,
    query_longitudes,
    BENCHMARK_ARTIFACTS_DIR / "nearby_semantic_top20_indices.npy",
    top_k=20,
    radius_km=100,
)

channels = {
    "local_e5": {"indices": local_e5, "scores": local_e5_scores},
    "local_tfidf": {"indices": local_tfidf, "scores": local_tfidf_scores},
    "nearby_e5": {"indices": nearby_e5},
    "global_e5": {"indices": global_e5, "scores": global_e5_scores},
}

pd.DataFrame({
    "channel": ["Local E5", "Local TF-IDF", "Nearby E5", "Global E5"],
    "candidates": [
        local_e5.shape[1],
        local_tfidf.shape[1],
        nearby_e5.shape[1],
        global_e5.shape[1],
    ],
    "coverage": [
        (local_e5[:, 0] >= 0).mean(),
        (local_tfidf[:, 0] >= 0).mean(),
        (nearby_e5[:, 0] >= 0).mean(),
        (global_e5[:, 0] >= 0).mean(),
    ],
})

In [ ]:
del tfidf_vectorizer, tfidf_item_matrix, tfidf_query_matrix
del item_embeddings, query_embeddings
_ = gc.collect()

## 5. Агрегация кандидатов

In [ ]:
model_path = ARTIFACTS_DIR / "final_linear_blender.joblib"
if not model_path.exists():
    raise FileNotFoundError(
        "Сначала выполните 02_pipeline_development.ipynb: "
        "он создаёт final_linear_blender.joblib"
    )

model_bundle = joblib.load(model_path)
if model_bundle["feature_names"] != FEATURE_NAMES:
    raise ValueError("Признаки модели не совпадают с текущей версией pipeline")

top50_positions = rank_with_model(
    model_bundle["model"],
    len(query_ids),
    channels,
    item_location_ids,
    query_location_ids,
    item_category_ids,
    query_category_ids,
    top_k=50,
)

pd.Series((top50_positions >= 0).sum(axis=1)).describe()

## 6. Формирование и проверка answer.csv

In [ ]:
predictions = positions_to_predictions(top50_positions, item_ids, top_k=50)
answer = build_answer(query_ids, predictions)
validate_answer(answer, query_ids, item_ids)

answer.head(3)

In [ ]:
answer_path = save_answer(answer, PROJECT_ROOT / "answer.csv")

saved_answer = pd.read_csv(
    answer_path,
    dtype={"query_id": "string", "answer": "string"},
    keep_default_na=False,
)
saved_predictions = validate_answer(saved_answer, query_ids, item_ids)

pd.Series({
    "path": str(answer_path),
    "rows": len(saved_answer),
    "unique_query_ids": saved_answer["query_id"].nunique(),
    "min_items": saved_predictions.map(len).min(),
    "max_items": saved_predictions.map(len).max(),
    "file_size_kb": answer_path.stat().st_size / 1024,
})